# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim, col
from pyspark.sql.types import StringType

# Read from bronze table

In [0]:
df = spark.table("workspace.bronze.erp_px_cat_g1v2")

# Transformations

- Trim text columns
- Change maintenance column data type to a boolean
- Rename column headers

### Trimming text columns

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Change maintenance column data to boolean

In [0]:
df = df.withColumn(
    "MAINTENANCE",
    F.when(F.col("MAINTENANCE") == "Yes", True)
     .when(F.col("MAINTENANCE") == "No", False)
     .otherwise(None)
)

### Renaming column headers

In [0]:
RENAME_MAPPING = {
    "ID": "category_id",
    "CAT": "category",
    "SUBCAT": "subcategory",
    "MAINTENANCE": "maintenance_flag"
}

for old_name, new_name in RENAME_MAPPING.items():
    df = df.withColumnRenamed(old_name, new_name)

### Sanity check dataframe

In [0]:
df.limit(10).display()

# Writing to silver table

In [0]:
df.write.mode("overwrite").saveAsTable("workspace.silver.erp_product_category")

### Sanity check silver table

In [0]:
%sql

SELECT * FROM workspace.silver.erp_product_category LIMIT 10